# Epic D04-E03: Exploratory Data Analysis (EDA)
## Product Category Profitability Study
**Objective:** Perform comprehensive statistical and graphical exploration of revenue distributions, cost structures, margin dispersions, discount dynamics, and regional performance.

---


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.sans-serif'] = 'Arial'

# Load clean dataset
df = pd.read_csv('../data/processed/cleaned_product_sales.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
print(f"Loaded {len(df):,} cleaned sales transactions.")

### 1. Revenue, Cost & Profit Comparison by Category

In [ ]:
# Aggregate financial metrics by product category
cat_perf = df.groupby('product_category').agg(
    total_revenue=('revenue', 'sum'),
    total_cost=('cost', 'sum'),
    total_profit=('profit', 'sum'),
    avg_margin=('profit_margin_pct', 'mean')
).sort_values('total_revenue', ascending=False).reset_index()

display(cat_perf)

# Grouped bar chart
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(cat_perf))
width = 0.28

ax.bar(x - width, cat_perf['total_revenue'] / 1e3, width, label='Net Revenue ($K)', color='#2b5c8f')
ax.bar(x, cat_perf['total_cost'] / 1e3, width, label='COGS ($K)', color='#d9534f')
ax.bar(x + width, cat_perf['total_profit'] / 1e3, width, label='Gross Profit ($K)', color='#5cb85c')

ax.set_xticks(x)
ax.set_xticklabels(cat_perf['product_category'], rotation=25, ha='right')
ax.set_ylabel('Amount ($ Thousands USD)')
ax.set_title('Category Financial Breakdown: Revenue vs Cost vs Profit')
ax.legend()
plt.tight_layout()
plt.show()

**Observation:**
* `Electronics` dominates top-line revenue (\$1.4M+) but carries a massive cost burden (\$1.3M), resulting in compressed margin share.
* `Furniture` exhibits extreme cost-to-revenue ratios, yielding almost zero cumulative profit despite substantial sales volume.
* `Apparel & Footwear` and `Home & Kitchen` generate high profit contributions relative to their top-line scale.

### 2. Profit Margin Distribution & Outliers (Boxplots)

In [ ]:
plt.figure(figsize=(11, 5.5))
order_cat = df.groupby('product_category')['profit_margin_pct'].median().sort_values(ascending=False).index

sns.boxplot(x='product_category', y='profit_margin_pct', data=df, order=order_cat, showmeans=True,
            meanprops={"marker":"o", "markerfacecolor":"red", "markeredgecolor":"black"})
plt.axhline(0, color='crimson', linestyle='--', label='Break-Even Point (0%)')
plt.title('Profit Margin (%) Distribution Across Categories (Sorted by Median)')
plt.xticks(rotation=25, ha='right')
plt.ylabel('Profit Margin (%)')
plt.legend()
plt.tight_layout()
plt.show()

### 3. The Discount Erosion Trap: Discount % vs Margin %

In [ ]:
plt.figure(figsize=(10, 6))
sample_df = df.sample(min(2000, len(df)), random_state=42)

sns.scatterplot(
    x=sample_df['discount_pct'] * 100,
    y=sample_df['profit_margin_pct'],
    hue=sample_df['product_category'],
    alpha=0.6,
    palette='tab10'
)
sns.regplot(
    x=df['discount_pct'] * 100,
    y=df['profit_margin_pct'],
    scatter=False,
    color='black',
    line_kws={'linestyle': '--', 'label': 'Margin Decay Trendline'}
)
plt.axhline(0, color='red', linestyle='-', label='Break-Even Threshold')
plt.title('Promotional Discount % vs Realized Profit Margin %')
plt.xlabel('Discount Rate Applied (%)')
plt.ylabel('Profit Margin (%)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 4. Correlation Heatmap Between Numerical Attributes

In [ ]:
plt.figure(figsize=(8, 6))
corr_cols = ['quantity', 'unit_price', 'unit_cost', 'discount_pct', 'revenue', 'cost', 'profit', 'profit_margin_pct']
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title('Correlation Matrix of Transaction Attributes')
plt.tight_layout()
plt.show()